### Import the necessary database

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [ ]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=4, cores=50, memory="200GB")

In [ ]:
# read all segmented trend patterns (L=10..73) into a dict of DataArrays
import os
dir_ICV_seg_TREND = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/'
len_segments = np.arange(10, 74, 1)

ds = {}
for L in len_segments:
    fpath = os.path.join(dir_ICV_seg_TREND, f"OBS_ICV_MK_trend_segments_L{L}.nc")
    da = xr.open_dataset(fpath, chunks={"lat": 10, "lon": 10})['trend']
    ds[f"ICV_trend_{L}yr"] = da


In [ ]:
ds

In [ ]:
# flip the longitude to -180 to 180
ds_adj = {}
for key in ds.keys():
    ds_adj[key] = preprosess.adjust_longitude(ds[key],ds[key].lon)

In [ ]:
ds_adj

In [ ]:
ds_temp = {}
for key in ds.keys():
    ds_temp[key] = xr.DataArray(
        ds_adj[key][0],
        coords={'segment': ds[key].segment, 'lat': ds[key].lat, 'lon': ds_adj[key][1]},
        dims=('segment', 'lat', 'lon')
    )

In [ ]:
ds_temp

In [ ]:
lat = ds_temp["ICV_trend_10yr"].lat
lon = ds_temp["ICV_trend_10yr"].lon
# Extratropical South Pacific region
# Arctic region 
lat1 = 66.5
lat2 = 90
lon1 = -180
lon2 = 180
# select the region

ds_Arctic_masked = {}
for key in ds.keys():
    ds_Arctic_masked[key] = data_process.selreg(ds_temp[key],lat, lon, lat1, lat2, lon1, lon2)

In [ ]:
ds_Arctic_masked

In [ ]:
# calculate the regional mean
ds_Arctic_mean = {}
for key in ds_Arctic_masked.keys():
    ds_Arctic_mean[key] = data_process.calc_weighted_mean(ds_Arctic_masked[key][0])

### specify the 5 and 95 percentile values for each year step and output the 10-73yr unforced percentile timeseires for each regions

In [ ]:
# define the function to calculate the percentile
def calc_percentile(da, q):
    """ Calculate the qth percentile of the data along the specified dimension.
    Args:
    da: xr.DataArray
    dim: str
    q: float
    Returns:
    xr.DataArray
    """
    # remove nans for da
    da = da.dropna(dim='segment')
    lower_percentile = np.percentile(da, q)
    upper_percentile = np.percentile(da, 100-q)
    
    return lower_percentile, upper_percentile

In [ ]:
# calculate the regional mean's percentile
# 5%---[0]
unforced_trend_arctic_lower_percentile = {}

# 95%---[1]
unforced_trend_arctic_upper_percentile = {}

for key in ds_Arctic_masked.keys():
    unforced_trend_arctic_lower_percentile[key], unforced_trend_arctic_upper_percentile[key] = calc_percentile(ds_Arctic_mean[key], 5)
    

In [ ]:
unforced_trend_arctic_lower_percentile

In [ ]:
# transform the dictionary to the dataarray
icv_years = list(unforced_trend_arctic_lower_percentile.keys())
unforced_trend_Arctic_lower_percentile_da = xr.DataArray(
	list(unforced_trend_arctic_lower_percentile.values()),
	coords={'ICV_trend_year_lower': icv_years},
	dims=['ICV_trend_year_lower'],
)
unforced_trend_Arctic_upper_percentile_da = xr.DataArray(
	list(unforced_trend_arctic_upper_percentile.values()),
	coords={'ICV_trend_year_upper': icv_years},
	dims=['ICV_trend_year_upper'],
)
    

In [ ]:
# save the percentile data
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG5/data/percentile/'
# use variable names that do not conflict with coordinate names
unforced_trend_Arctic_lower_percentile_da.to_dataset(name="ICV_trend_lower").to_netcdf(dir_out+'internal_Arctic_trend_lower_percentile.nc')
unforced_trend_Arctic_upper_percentile_da.to_dataset(name="ICV_trend_upper").to_netcdf(dir_out+'internal_Arctic_trend_upper_percentile.nc')
